# Event IC-optimization — package workflow demo

Runs the full W&DL storyline pipeline for ONE event entirely through `heatwave_ic` function calls — no copy-pasted pipeline code. To run a different event (e.g. the PNW 2021 reproduction), change **only** the config path in the next cell.

Needs a GPU + GCS access → run on **Colab**, not the local CPU venv.

In [3]:
CONFIG = "configs/stjohns_aug2025.yaml"   # <-- the ONLY per-event change
REPO_URL = "https://github.com/ieadoboe/heatwave-initial-conditions.git"

import sys
from pathlib import Path

# Find the repo root (idempotent: works on re-runs and local checkouts).
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "heatwave_ic").is_dir()), None)
if root is None and "google.colab" in sys.modules:
    !git clone {REPO_URL}
    root = Path.cwd() / Path(REPO_URL).stem
if root is None or not (root / "heatwave_ic").is_dir():
    raise RuntimeError(
        "heatwave_ic/ not found. Either the clone failed, or the package "
        "hasn't been pushed to GitHub yet (git push from the local repo)."
    )
%cd {root}
sys.path.insert(0, str(root))
if "google.colab" in sys.modules:
    %pip install -q -U neuralgcm dinosaur gcsfs optax tqdm pyyaml zarr

Cloning into 'heatwave-initial-conditions'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 166 (delta 11), reused 50 (delta 11), pack-reused 112 (from 1)
Receiving objects: 100% (166/166), 168.61 MiB | 19.28 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/heatwave-initial-conditions/heatwave-initial-conditions


In [4]:
from heatwave_ic import (
    load_config, describe, load_model, build_ic_zarr, load_ic_on_model_grid,
    optimize_event, box_t1000_trajectory, make_run_dir,
    save_losses, save_state_fields, save_trajectory_nc, plots,
)

cfg = load_config(CONFIG)
print(describe(cfg))

FileNotFoundError: [Errno 2] No such file or directory: 'configs/stjohns_aug2025.yaml'

In [ ]:
model = load_model(cfg["model_name"])
build_ic_zarr(model, cfg)                       # no-op if already built
eval_era5 = load_ic_on_model_grid(model, cfg["paths"]["ic_zarr"])

## Optimize

`optimize_event` uses the config's (tuned) hyperparameters; keyword overrides sweep them without re-editing the YAML, e.g.
```python
optimize_event(model, eval_era5, cfg, beta=20, lam=30, iterations=30)
optimize_event(model, eval_era5, cfg, weights={"specific_humidity": 10})
```

In [ ]:
result = optimize_event(model, eval_era5, cfg)
print(f"final loss {result['losses'][-1]:.4f}   "
      f"final box T {result['box_T_K'][-1] - 273.15:.2f} C   "
      f"IC violation {result['reg'][-1]:.4g}")

In [ ]:
event, run = cfg["event"], cfg["run"]

traj_ori = box_t1000_trajectory(
    model, result["initial_state"], result["all_forcings"],
    result["outer_steps"], result["lat_i"], result["lon_i"], run["init_date"])
traj_opt = box_t1000_trajectory(
    model, result["optimized_state"], result["all_forcings"],
    result["outer_steps"], result["lat_i"], result["lon_i"], run["init_date"])

print(f"storyline gain: +{traj_opt.max() - traj_ori.max():.2f} C at the peak")

plots.plot_loss(result, title=f"{event['name']} --- loss",
                save=f"plots/{event['name']}_opt_loss.pdf")
plots.plot_storyline(traj_ori, traj_opt, event["start"], event["end"],
                     title=f"{event['name']} storyline",
                     save=f"plots/{event['name']}_storyline.pdf");

In [ ]:
out_dir = make_run_dir(cfg, result["params"])
save_losses(result, out_dir)
save_trajectory_nc(model, result["optimized_state"], result["all_forcings"],
                   result["outer_steps"], f"{out_dir}/optimized.nc")
save_trajectory_nc(model, result["initial_state"], result["all_forcings"],
                   result["outer_steps"], f"{out_dir}/original.nc")
save_state_fields(model, result["optimized_state"], out_dir, "opt")
save_state_fields(model, result["initial_state"], out_dir, "original")
print(f"outputs -> {out_dir}")

CLI equivalent of this whole notebook:
```bash
python scripts/optimize_event.py --config configs/stjohns_aug2025.yaml --build-ic
python scripts/optimize_event.py --config configs/stjohns_aug2025.yaml
```